# Assignment 1


In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import urllib.request

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print("device:", device)

SEQ_LEN = 24
BATCH_SIZE = 128
EMBED_DIM = 32
HIDDEN_DIM = 64
LR = 0.002
EPOCHS = 3
NUM_LAYERS = 1

url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
text = urllib.request.urlopen(url).read().decode("utf-8")
words = text.split()

# need these for eval 2 or crashes
homework_words = ["polish", "nail", "mirror", "red"]
vocab = sorted(set(w.lower() for w in words + homework_words))

stoi = {w: i for i, w in enumerate(vocab)}
itos = {i: w for w, i in stoi.items()}
vocab_size = len(stoi)
print("vocab_size:", vocab_size)

/Users/joe/Desktop/code/schoolcode/GitHub_School/genai/.venv/lib/python3.13/site-packages/torch/_subclasses/functional_tensor.py:283: UserWarning: Failed to initialize NumPy: No module named 'numpy' (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/torch/csrc/utils/tensor_numpy.cpp:84.)
  cpu = _conversion_method_template(device=torch.device("cpu"))


device: mps
vocab_size: 23643


In [2]:
# dataloader

word_ids = [stoi[w.lower()] for w in words]

class SeqDataset(Dataset):
    def __init__(self, data, seq_len):
        self.data = data
        self.seq_len = seq_len
        
    def __len__(self):
        return max(0, len(self.data) - self.seq_len)
        
    def __getitem__(self, i):
        chunk = self.data[i : i + self.seq_len + 1]
        x = torch.tensor(chunk[:-1], dtype=torch.long)
        y = torch.tensor(chunk[1:], dtype=torch.long)
        return x, y

train_ds = SeqDataset(word_ids, SEQ_LEN)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)

## 1. UniLM


In [3]:
class UniLM(nn.Module):
    def __init__(self):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, EMBED_DIM)
        self.lstm = nn.LSTM(EMBED_DIM, HIDDEN_DIM, NUM_LAYERS, batch_first=True)
        self.head = nn.Linear(HIDDEN_DIM, vocab_size)
        
    def forward(self, x, hidden=None):
        e = self.embed(x)
        out, (h, c) = self.lstm(e, hidden)
        return self.head(out), (h, c)

    def get_context(self, x):
        e = self.embed(x)
        out, _ = self.lstm(e)
        return out

uni_lm = UniLM().to(device)
opt = torch.optim.Adam(uni_lm.parameters(), lr=LR)

print("Training...")
for epoch in range(EPOCHS):
    uni_lm.train()
    total_loss = 0
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        logits, _ = uni_lm(x)
        loss = nn.functional.cross_entropy(logits.reshape(-1, vocab_size), y.reshape(-1))
        
        opt.zero_grad()
        loss.backward()
        opt.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1} Loss: {total_loss/len(train_loader):.4f}")

Training...
Epoch 1 Loss: 6.5538
Epoch 2 Loss: 5.2609
Epoch 3 Loss: 4.6069


## 2. BiLM (ELMo)

In [4]:
class BiLM(nn.Module):
    def __init__(self):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, EMBED_DIM)
        # Two separate LSTMs
        self.fwd_lstm = nn.LSTM(EMBED_DIM, HIDDEN_DIM, NUM_LAYERS, batch_first=True)
        self.bwd_lstm = nn.LSTM(EMBED_DIM, HIDDEN_DIM, NUM_LAYERS, batch_first=True)
        self.head_fwd = nn.Linear(HIDDEN_DIM, vocab_size)
        self.head_bwd = nn.Linear(HIDDEN_DIM, vocab_size)

    def forward(self, x):
        e = self.embed(x)
        e_rev = torch.flip(e, dims=[1])
        
        out_fwd, _ = self.fwd_lstm(e)
        out_bwd, _ = self.bwd_lstm(e_rev)
        
        out_bwd = torch.flip(out_bwd, dims=[1])
        
        return self.head_fwd(out_fwd), self.head_bwd(out_bwd)

    def get_context(self, x):
        e = self.embed(x)
        e_rev = torch.flip(e, dims=[1])
        
        out_fwd, _ = self.fwd_lstm(e)
        out_bwd, _ = self.bwd_lstm(e_rev)
        out_bwd = torch.flip(out_bwd, dims=[1])
        
        return torch.cat([out_fwd, out_bwd], dim=-1)

bi_lm = BiLM().to(device)
opt = torch.optim.Adam(bi_lm.parameters(), lr=LR)

print("Training...")
for epoch in range(EPOCHS):
    bi_lm.train()
    loss_sum = 0
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        
        logits_f, logits_b = bi_lm(x)
        
        l1 = nn.functional.cross_entropy(logits_f.reshape(-1, vocab_size), y.reshape(-1))
        
        y_rev = torch.flip(y, dims=[1])
        l2 = nn.functional.cross_entropy(logits_b.reshape(-1, vocab_size), y_rev.reshape(-1))
        
        loss = l1 + l2
        
        opt.zero_grad()
        loss.backward()
        opt.step()
        loss_sum += loss.item()

    print(f"Epoch {epoch+1} Loss: {loss_sum/len(train_loader):.4f}")

Training...
Epoch 1 Loss: 13.9597
Epoch 2 Loss: 12.3995
Epoch 3 Loss: 11.3571


## 3. Seq2Seq

In [5]:
class Seq2Seq(nn.Module):
    def __init__(self):
        super().__init__()
        self.enc_embed = nn.Embedding(vocab_size, EMBED_DIM)
        self.dec_embed = nn.Embedding(vocab_size, EMBED_DIM)
        self.encoder = nn.LSTM(EMBED_DIM, HIDDEN_DIM, NUM_LAYERS, batch_first=True)
        self.decoder = nn.LSTM(EMBED_DIM, HIDDEN_DIM, NUM_LAYERS, batch_first=True)
        self.head = nn.Linear(HIDDEN_DIM, vocab_size)

    def forward(self, src, tgt):
        e_src = self.enc_embed(src)
        _, (h, c) = self.encoder(e_src)
        
        e_tgt = self.dec_embed(tgt)
        out, _ = self.decoder(e_tgt, (h, c))
        
        return self.head(out)

    def get_context(self, src):
        e = self.enc_embed(src)
        out, _ = self.encoder(e)
        return out

seq2seq = Seq2Seq().to(device)
opt = torch.optim.Adam(seq2seq.parameters(), lr=LR)

print("Training...")
for epoch in range(EPOCHS):
    seq2seq.train()
    loss_sum = 0
    for x, _ in train_loader:
        x = x.to(device)
        
        # Reverse
        tgt = torch.flip(x, dims=[1])
        
        dec_in = tgt[:, :-1]
        dec_out = tgt[:, 1:]
        
        logits = seq2seq(x, dec_in)
        loss = nn.functional.cross_entropy(logits.reshape(-1, vocab_size), dec_out.reshape(-1))
        
        opt.zero_grad()
        loss.backward()
        opt.step()
        loss_sum += loss.item()

    print(f"Epoch {epoch+1} Loss: {loss_sum/len(train_loader):.4f}")

Training...
Epoch 1 Loss: 6.4099
Epoch 2 Loss: 4.9223
Epoch 3 Loss: 4.0952


## Eval 1: Static embedding neighbors

In [6]:
def topk_cosine(embed, word, k=8):
    idx = stoi.get(word.lower())
    if idx is None:
        return []
        
    w = embed.weight.detach()
    v = w[idx].unsqueeze(0)
    
    sim = torch.nn.functional.cosine_similarity(v, w, dim=1)
    
    vals, indices = torch.topk(sim, k + 1)
    
    out = []
    for i in range(1, len(indices)):
        name = itos[indices[i].item()]
        score = vals[i].item()
        out.append((name, round(score, 4)))
    return out

eval_words = ["love", "death", "man", "woman"]
models = [
    ("UniLM", uni_lm.embed), 
    ("BiLM", bi_lm.embed), 
    ("Seq2Seq", seq2seq.enc_embed)
]

print("Eval 1: static neighbors")
for name, emb in models:
    print(name)
    for w in eval_words:
        print(f"  {w}: {topk_cosine(emb, w)}")
    print()

Eval 1: static neighbors
UniLM
  love: [('vienna,', 0.7644), ('prince', 0.6756), ('about!', 0.6034), ('cut', 0.6005), ('elbow;', 0.5931), ('vain.', 0.5825), ('lead', 0.568), ('devil,', 0.5634)]
  death: [("arm'd,", 0.6497), ('gift', 0.6021), ('sort,', 0.5804), ('brother;', 0.5566), ('circumstances', 0.5557), ('foot,', 0.5551), ('fired', 0.5482), ("upon's;", 0.5467)]
  man: [('weeping.', 0.6487), ('clay.', 0.6109), ("punish'd,", 0.6001), ('mads', 0.5959), ('us', 0.5824), ('manner', 0.5801), ('jewel', 0.5788), ('souls', 0.5711)]
  woman: [('metal', 0.7506), ('promises', 0.7144), ('conference', 0.6668), ('daintry,', 0.6656), ("ability's", 0.6456), ('suffered', 0.6269), ('plainer', 0.6184), ('waves,', 0.615)]

BiLM
  love: [("overta'en", 0.6148), ('apprenticehood', 0.6081), ('insufficience,', 0.6007), ('stick', 0.5883), ('appears', 0.5861), ('foe,', 0.5682), ('revenue', 0.5645), ('threatened', 0.5637)]
  death: [('belman', 0.7268), ('observer', 0.6902), ('spur?', 0.6461), ('day', 0.6448), 

## Eval 2: Contextual word similarity

In [11]:
def sent_to_tensor(s):
    ids = [stoi[w.lower()] for w in s.split()]
    return torch.tensor([ids], dtype=torch.long).to(device)

s1 = sent_to_tensor("i polish the mirror")
s2 = sent_to_tensor("the nail polish is red")

p1 = 1 # index of polish for s1
p2 = 2 # index of polish for s2

def get_vec(model, x, pos):
    model.eval()
    h = model.get_context(x)
    return h[0, pos].detach().unsqueeze(0)

# Get all vectors
v1_u = get_vec(uni_lm, s1, p1)
v2_u = get_vec(uni_lm, s2, p2)

v1_b = get_vec(bi_lm, s1, p1)
v2_b = get_vec(bi_lm, s2, p2)

v1_s = get_vec(seq2seq, s1, p1)
v2_s = get_vec(seq2seq, s2, p2)

# Print results
print("Eval 2: polish similari")
print("UniLM:", torch.nn.functional.cosine_similarity(v1_u, v2_u).item())
print("BiLM:",  torch.nn.functional.cosine_similarity(v1_b, v2_b).item())
print("Seq2Seq:", torch.nn.functional.cosine_similarity(v1_s, v2_s).item())

Eval 2: polish similari
UniLM: 0.853613018989563
BiLM: 0.886400580406189
Seq2Seq: 0.4951239228248596


## Q1 - For each model, state exactly which tensor you used as:1.the static embedding (Eval 1)2.the contextual embedding (Eval 2)Be specific (e.g., embedding table, hidden state, encoder output)

"the embedding" refers to 2 different tensors sources depending on the eval. For eval 1, I used the static embedding lookup table = model.embed.weight for UniLM and BiLM and model.enc_embed.weight for Seq2Seq. This table represents words independent of context, which is why "love" in the UniLM always had "vienna" with cosine 0.7644 as its nearest neighbor regardless of where it appeared. For Eval 2, I used the LSTM hidden states. For the UniLM, this was the hidden state at the specific token position, for BiLM, the forward and backward hidden states concatenated to capture the full context, and for Seq2Seq, the encoder's output hidden state. 

## Q2 - Explain why the cosine similarity for “polish” changes (or does not change) between the unidirectional  LM  and  the  bidirectional  LM.Tie  your  explanation  directly  to  the  training objective and available context

The cosine similarity for "polish" was 0.8536 for the UniLM and 0.8864 for the BiLM. I expected the BiLM score to be lower because it sees the future context ("is red") which clarifies that "polish" is a noun, whereas the UniLM only sees the past. However, because "polish" was manually added to the vocabulary and not actually present in the Tiny Shakespeare, the model didnt learn strong contextual cues for it which explains why both models produced high scores since they were relying on the random embeddings rather than learned training.

## Q3 - Explain  why  the  Seq2Seq  encoder  representation  behaves  differently  from  the  LM representations,   even   though   all   models   use   LSTMs.Focus   on   conditional   vs unconditional modeling

The Seq2Seq score was significantly lower at 0.4951, compared to 0.8536 (UniLM) and 0.8864 (BiLM). UniLM and BiLM are unconditional models. They just predict the probability of the next word. In contrast, the Seq2Seq encoder is part of a conditional thinking so its job is to compress the entire source sentence into a single context vector (the final hidden state) that the decoder conditions on to generate the reverse sequence. This difference in objective forces the encoder to learn a completely different type of representation in this case for "polish" which is like the cosine score is lower.

## Q4 - Make one of the following changes and rerun Eval 2:•remove BOS/EOS tokens in Seq2Seq•reduce sequence length by half•use the embedding table instead of hidden statesState what changed and why


In [10]:
def get_static(model, word):
    table = model.enc_embed if hasattr(model, 'enc_embed') else model.embed
    idx = stoi[word]
    return table.weight[idx].detach().unsqueeze(0)

v_uni = get_static(uni_lm, "polish")
v_bi = get_static(bi_lm, "polish")
v_seq = get_static(seq2seq, "polish")

print("q4 eval (static embeddings):")
print(f"unilm: {torch.nn.functional.cosine_similarity(v_uni, v_uni).item():.4f}")
print(f"bilm: {torch.nn.functional.cosine_similarity(v_bi, v_bi).item():.4f}")
print(f"seq2seq: {torch.nn.functional.cosine_similarity(v_seq, v_seq).item():.4f}")

q4 eval (static embeddings):
unilm: 1.0000
bilm: 1.0000
seq2seq: 1.0000


after changing it to use the static embedding table instead of the hidden states for Eval 2. When I reran the code with this change, the cosine similarity for all three models became exactly 1.0. This happened because the static embedding table is just a fixed lookup dictionary that does not care about the surrounding words. This shows that the differences saw earlier like the 0.49 score for Seq2Seq were caused entirely by the LSTM layers processing the context, since the underlying word vectors for "polish" are identical in the actual embedding table.